# 8. 텍스트 수집, 전처리와 토큰화

| 순서 | 내용 |
|------|------|
| 1 | 텍스트마이닝 개요 |
| 2 | 텍스트 데이터 수집: 웹 크롤링 |
| 3 | 정규표현식 |
| 4 | 데이터 정제 |
| 5 | 토큰화와 형태소 분석 |
| 6 | 리뷰 키워드 확인 |


# 텍스트마이닝 개요

## 1. 텍스트마이닝이란?

### 1.1 텍스트마이닝의 정의
- 비정형 텍스트에서 유용한 정보, 패턴, 주제, 감정, 관계를 추출하는 데이터 분석 방법입니다.
- 자연어 처리(NLP)는 컴퓨터가 언어를 다루게 하는 기술 영역이고, 텍스트마이닝은 그 기술을 활용해 비즈니스·사회·연구 문제를 분석하는 응용 흐름에 가깝습니다.
- 실무에서는 `수집 -> 정제 -> 토큰화 -> 벡터화/임베딩 -> 분석/모델링 -> 해석` 순서로 진행되는 경우가 많습니다.

### 1.2 일반적인 데이터 분석과의 차이
| 구분 | 정형 데이터 분석 | 텍스트마이닝 |
|------|------------------|--------------|
| 원본 형태 | 숫자, 범주, 날짜처럼 열 구조가 명확함 | 문장, 댓글, 기사, 문서처럼 자유롭게 작성됨 |
| 주요 전처리 | 결측치, 이상치, 스케일 조정, 인코딩 | 불필요한 문자 제거, 표기 통일, 토큰화, 형태소 분석 |
| 특징 생성 | 기존 열을 변환하거나 조합 | 단어, n-gram, 키워드, 문서 벡터, 임베딩 생성 |
| 해석 관점 | 변수와 목표값의 관계 | 단어·문맥·주제·감정·의견의 흐름 |
| 어려운 점 | 데이터 품질, 변수 선택, 모델 평가 | 중의성, 신조어, 오타, 문맥, 도메인 용어 |

### 1.3 주요 활용 사례
- **리뷰 분석**: 상품·서비스 리뷰에서 만족 요인, 불만, 개선 요구, 감성 점수를 찾습니다.
- **뉴스 분석**: 기사 제목과 본문을 모아 이슈 흐름, 언론사별 관점, 주요 인물·기업·사건을 추적합니다.
- **여론 분석**: 댓글, 게시글, 설문 응답에서 찬반 의견, 관심 주제, 정책 반응을 파악합니다.
- **키워드 분석**: 빈도, TF-IDF, 동시출현, 워드클라우드로 문서 집합의 핵심 단어를 요약합니다.
- **챗봇/RAG**: 문서를 잘게 나누고 임베딩으로 검색한 뒤, 관련 근거를 언어모델에 전달해 답변을 생성합니다.

### 1.4 기본 처리 흐름
텍스트 분석은 보통 `수집 -> 정제 -> 토큰화 -> 수치화 -> 분석` 순서로 진행합니다. 이 노트북에서는 앞의 세 단계인 수집, 정제, 토큰화를 다룹니다.


## 2. 텍스트 데이터 다루기

### 2.1 텍스트 데이터 수집: 웹 크롤링

텍스트마이닝은 분석할 텍스트를 확보하는 일에서 시작합니다. 직접 설문을 만들 수도 있지만, 뉴스·리뷰·게시글처럼 이미 웹에 공개된 텍스트를 수집해서 분석하는 경우도 많습니다.

이번 장에서는 **Playwright**로 브라우저를 직접 열고, 화면 안의 요소를 선택해 텍스트를 수집하는 방법을 살펴봅니다. Playwright를 사용하면 실제 브라우저 동작, 기다림, iframe 선택자를 함께 다룰 수 있습니다.

데이터 분석 흐름에서 크롤링은 다음 단계의 가장 앞에 있습니다.

`수집 -> 정제 -> 토큰화 -> 벡터화 -> 모델링`

<img src="image/web_crawling_background.svg" width="780">


#### 웹에서 텍스트를 가져올 때 보는 기본 요소

웹 크롤링을 처음 할 때는 모든 웹 기술을 다 이해할 필요가 없습니다.  
이번 장에서는 **주소(URL)를 열고, 화면에 있는 표나 목록에서 필요한 글자를 가져오는 것**에 집중합니다.

| 요소 | 역할 | 크롤링에서 확인할 점 |
|------|------|----------------------|
| 브라우저 | 웹 페이지를 열어 화면에 보여줌 | 개발자도구로 제목, 날짜, 표 위치를 확인함 |
| URL | 웹 페이지의 주소 | 검색어, 카테고리, 페이지 번호처럼 바뀌는 값을 찾음 |
| HTTP/HTTPS | 브라우저와 서버가 데이터를 주고받는 방식 | 페이지가 정상으로 열리는지 확인함 |
| 웹 서버 | 요청받은 페이지나 데이터를 보내줌 | 같은 주소를 코드에서도 열 수 있음 |
| HTML | 페이지의 내용과 구조 | 제목, 표, 링크, 날짜가 들어 있는 태그를 찾음 |
| CSS | 화면 스타일을 지정하는 정보 | `class` 이름이 선택자 힌트가 되기도 함 |
| JavaScript | 화면을 나중에 채우거나 버튼 동작을 처리함 | 데이터가 늦게 보이면 기다림이 필요할 수 있음 |

예를 들어 아래 주소에서는 `q=python`이 검색어, `page=1`이 페이지 번호입니다.

```text
https://example.com/search?q=python&page=1
```

텍스트 수집에서는 이런 값을 바꿔가며 여러 페이지의 제목, 본문 요약, 날짜, 링크를 모을 수 있습니다.


#### HTML 태그 읽기

HTML은 웹 페이지의 뼈대입니다. 태그가 중첩되면서 문서 구조를 만들고, 각 태그 안에 텍스트나 링크가 들어갑니다.

```html
<tr class="price-row">
  <td class="date">2026.05.17</td>
  <td class="close">70,000</td>
  <td class="volume">12,345,678</td>
</tr>
```

위 HTML에서 `tr`은 표의 한 행, `td`는 표의 한 칸입니다.  
`class="date"`처럼 태그에 붙은 정보는 **속성(attribute)** 입니다.

크롤링은 결국 화면 안에서 **반복되는 구조**를 찾는 일입니다.  
뉴스 목록은 여러 행이 반복되고, 각 행 안에 제목, 언론사, 날짜, 링크가 들어 있습니다.

Playwright에서는 이런 반복 요소를 `locator()`로 찾습니다.

```python
rows = page.locator("tr.price-row")
first_row = rows.nth(0)
date = await first_row.locator("td.date").inner_text()
```

처음에는 선택자를 완벽히 외우기보다, 개발자도구에서 반복되는 행과 그 안의 제목 태그를 찾는 연습에 집중하면 됩니다.


#### 개발자도구로 수집 위치 찾기

크롤링 코드를 바로 작성하기보다 브라우저 개발자도구로 먼저 확인하면 시행착오가 줄어듭니다. Chrome 기준으로는 페이지에서 마우스 오른쪽 클릭 후 **검사**를 누르면 됩니다.

이번 장에서는 주로 **Elements 탭**만 확인합니다.

- 화면에 보이는 글자가 HTML의 어느 태그에 들어 있는지 확인합니다.
- 제목, 날짜, 표의 행처럼 반복되는 단위를 찾습니다.
- 태그 이름, `class`, `id`를 보고 선택자 힌트를 얻습니다.
- 목록에서 한 건을 나타내는 반복 단위와 그 안의 제목·날짜·링크 위치를 확인합니다.

개발자도구에서 CSS selector를 복사할 수도 있지만, 너무 긴 selector는 페이지가 조금만 바뀌어도 깨질 수 있습니다.  
`body > div > table > tbody > tr:nth-child(3)`처럼 위치에 의존하는 선택자보다 `table.type2 tr`, `td.title a.tit`처럼 의미 있는 태그와 클래스 조합을 쓰는 편이 안정적입니다.


#### 데이터 수집 원리

크롤링은 사람이 브라우저로 하는 일을 코드로 반복하는 과정입니다.

<img src="image/web_crawling_flow.svg" width="760">

기본 순서는 다음과 같습니다.

1. 수집할 항목을 정합니다. 예: 제목, 본문 요약, 날짜, 링크
2. 브라우저 개발자도구나 Playwright로 반복되는 HTML 구조를 확인합니다.
3. `page.goto()`로 수집할 페이지를 엽니다.
4. `locator()`로 반복 항목과 제목·본문·날짜 위치를 찾습니다.
5. `count()`, `nth()`, `inner_text()`, `get_attribute()`로 값을 하나씩 꺼냅니다.
6. 페이지 번호, 검색어, 카테고리 같은 값을 바꿔가며 반복 수집합니다.
7. 누락값, 중복, 날짜 범위를 확인한 뒤 데이터프레임이나 CSV로 저장합니다.

이번 장에서는 최신 웹 기술을 깊게 다루기보다, **화면에 보이는 표와 목록에서 텍스트를 가져오는 감각**을 익히는 데 집중합니다.

| 도구 | 적합한 상황 | 장점 | 주의할 점 |
|------|-------------|------|-----------|
| `Playwright` | 브라우저로 페이지를 열고 표나 목록을 확인해야 할 때 | 실제 브라우저처럼 실행되고 기다림 처리가 편함 | 최초 브라우저 설치가 필요함 |

크롤링 코드로 정리할 때는 다음 요소를 확인합니다.

- `page.goto(...)`: 수집할 페이지로 이동
- `locator(...)`: 반복되는 항목과 필요한 하위 요소 선택
- `inner_text()`: 화면에 보이는 텍스트 가져오기
- `get_attribute("href")`: 링크 주소 가져오기
- `wait_for()`: 데이터가 나타날 때까지 기다리기

크롤링할 때는 사이트의 `robots.txt`와 이용약관을 확인하고, 너무 빠르게 반복 요청하지 않아야 합니다. 개인정보, 저작권, 유료 콘텐츠도 함부로 수집하거나 재배포하면 안 됩니다.


#### Playwright로 웹 텍스트 수집하기

고정된 사이트의 완성 코드를 제공하기보다, 수업 중 선택한 사이트를 보면서 수집 절차를 직접 따라갑니다. 사이트마다 URL 구조, HTML 태그, iframe 여부, 로딩 방식이 다르기 때문에 **함수의 역할과 선택자 확인 방법**을 먼저 이해하는 것이 중요합니다.

Playwright 수집은 보통 다음 흐름으로 진행합니다.

1. 수집할 공개 페이지와 수집 항목을 정합니다.
2. 개발자도구에서 반복되는 HTML 구조를 찾습니다.
3. Playwright로 페이지를 열고 필요한 요소를 선택합니다.
4. 텍스트, 링크, 날짜 같은 값을 꺼냅니다.
5. 행 단위로 모아 `pandas.DataFrame`으로 만들고 CSV로 저장합니다.
6. 저장한 원본 표는 2.3 데이터 정제 단계에서 중복, 결측, 공백, HTML 흔적을 정리합니다.


#### Playwright 주요 함수와 사용법

| 목적 | 함수 또는 메서드 | 사용 예 | 설명 |
|------|------------------|---------|------|
| Playwright 시작 | `async_playwright().start()` | `playwright = await async_playwright().start()` | 브라우저 자동화를 시작합니다. |
| 브라우저 열기 | `playwright.chromium.launch()` | `browser = await playwright.chromium.launch(headless=False)` | Chromium 브라우저를 엽니다. `headless=False`이면 창이 보입니다. |
| 새 페이지 만들기 | `browser.new_page()` | `page = await browser.new_page(locale="ko-KR")` | 새 탭을 만들고 언어 같은 옵션을 지정합니다. |
| URL 이동 | `page.goto()` | `await page.goto(url, wait_until="load")` | 지정한 주소로 이동합니다. |
| 요소 선택 | `page.locator()` | `items = page.locator(".list-item")` | CSS 선택자로 화면 요소를 찾습니다. |
| iframe 내부 선택 | `page.frame_locator()` | `frame = page.frame_locator("iframe")` | iframe 안에 있는 요소를 선택할 때 사용합니다. |
| 요소 개수 확인 | `locator.count()` | `n = await items.count()` | 선택된 요소가 몇 개인지 확인합니다. |
| n번째 요소 선택 | `locator.nth()` | `item = items.nth(0)` | 반복 요소 중 특정 순서의 요소를 고릅니다. |
| 보이는 텍스트 추출 | `locator.inner_text()` | `text = await item.inner_text()` | 화면에 보이는 텍스트를 가져옵니다. |
| 속성 추출 | `locator.get_attribute()` | `href = await link.get_attribute("href")` | 링크 주소처럼 태그 속성값을 가져옵니다. |
| 로딩 대기 | `locator.wait_for()` | `await items.first.wait_for(timeout=10000)` | 요소가 나타날 때까지 기다립니다. |
| 짧은 간격 두기 | `page.wait_for_timeout()` | `await page.wait_for_timeout(500)` | 반복 수집 사이에 잠시 기다립니다. |
| 종료 | `browser.close()` / `playwright.stop()` | `await browser.close()` | 수집이 끝난 뒤 브라우저와 Playwright를 종료합니다. |

노트북에서는 비동기 실행 방식 때문에 수업 중 별도의 실행 보조 함수가 제공될 수 있습니다. 핵심은 `goto -> locator -> inner_text/get_attribute -> DataFrame 저장` 흐름입니다.


#### 수집 설계 체크리스트

코드를 작성하기 전에 아래 항목을 먼저 정리하면 시행착오가 줄어듭니다.

| 점검 항목 | 확인 질문 | 기록 예시 |
|-----------|-----------|-----------|
| 수집 대상 | 어떤 페이지에서 무엇을 가져올 것인가? | 뉴스 제목, 리뷰 본문, 상품명, 날짜 |
| URL 패턴 | 검색어, 페이지 번호, 카테고리 값이 URL에서 바뀌는가? | `page=1`, `query=검색어` |
| 반복 단위 | 한 건을 나타내는 HTML 반복 구조는 무엇인가? | `.item`, `li`, `tr` |
| 세부 선택자 | 제목, 날짜, 링크는 반복 단위 안의 어디에 있는가? | `.title`, `.date`, `a` |
| 페이지 이동 | 다음 페이지를 URL로 바꾸는가, 버튼을 클릭하는가? | URL 변경, `click()` 필요 |
| 저장 열 | 최종 표에 어떤 열을 남길 것인가? | `title`, `text`, `date`, `link`, `page` |
| 품질 확인 | 빈 값, 중복, 광고 문구가 섞이는가? | 결측 개수, 중복 행 수 |

수집 결과는 먼저 원본 형태로 저장합니다. 예를 들어 `raw_collected_text.csv`처럼 저장해 두면, 이후 정제 규칙을 바꾸더라도 원본을 다시 수집하지 않아도 됩니다.


#### 개별 수집 가이드

각자 원하는 공개 사이트를 하나 정해 Playwright로 텍스트를 수집하고, 표 형태로 저장합니다.

| 요구사항 | 내용 |
|----------|------|
| 수집 대상 | 로그인 없이 볼 수 있는 공개 페이지 |
| 수집 규모 | 최소 20개 이상의 텍스트 행 |
| 필수 열 | 텍스트 열 1개 이상, 출처 또는 링크 열 1개 이상 |
| 권장 열 | `title`, `text`, `date`, `link`, `source`, `page` |
| 저장 형식 | `pandas.DataFrame`으로 만든 뒤 CSV 저장 |
| 제출 기준 | 원본 CSV와 수집 항목 설명을 함께 정리 |

주의할 점:
- 사이트의 이용약관과 `robots.txt`를 확인합니다.
- 로그인, 유료 콘텐츠, 개인정보가 필요한 페이지는 피합니다.
- 반복 요청 사이에는 짧은 대기 시간을 둡니다.
- 수집한 원본 표는 바로 분석하지 말고, 다음 단계에서 결측치, 중복, 공백, HTML 흔적을 정제합니다.


### 2.2 웹 크롤링 실습

각자 원하는 사이트의 데이터를 수집 후 csv 파일로 저장합니다.  
저장한 데이터를 아래에서 배우는 기술들을 활용해 분석할 예정입니다.

### 2.3 정규표현식(Regex)
정규표현식(Regular Expression, **regex**)은 **문자열에서 특정 패턴을 찾고/바꾸고/분리**하는 강력한 도구입니다.  
전자영수증에서 숫자만 뽑아내고, 로그에서 IP만 추출하고, 텍스트 노이즈를 빠르게 정리하는 등 **대량의 텍스트 처리 자동화**에 핵심적으로 쓰입니다.


#### 왜 Regex를 쓰나요?
- **일관된 패턴**을 한 번에 처리 (이메일, URL, 날짜, 숫자, 해시태그 등)
- **간결한 코드**로 복잡한 문자열 조작 수행
- 전처리(클리닝) 단계에서 **재사용 가능한 규칙**으로 품질 유지


#### 핵심 문법(요약)
- **문자클래스**: `\d`(숫자), `\D`(숫자 아님), `\w`(단어문자: [A-Za-z0-9_]), `\s`(공백)  
- **반복/수량자**: `*`(0+), `+`(1+), `?`(0 or 1), `{m,n}`(m~n회)  
- **그룹/선택**: `( )`(캡처 그룹), `(?: )`(비캡처), `|`(OR)  
- **앵커**: `^`(문자열/행 시작), `$`(문자열/행 끝), `\b`(단어 경계)  
- **플래그**: `re.I`(대소문자 무시), `re.M`(멀티라인: ^,$를 행 단위로), `re.S`(dot이 개행 포함)  
- **탐욕/게으름**: `.*`(탐욕적), `.*?`(게으름; 가능한 한 짧게)

> **주의**: 파이썬에서는 `r"..."` **원시 문자열**을 사용해 백슬래시 이스케이프를 피하세요.


#### 정규표현식 주요 패턴 표

| 패턴 | 의미 | 예시 | 매칭 결과 |
|------|------|------|-----------|
| `.` | 임의의 한 문자(개행 제외) | `a.c` | `abc`, `axc` |
| `^` | 문자열/행의 시작 | `^Hi` | `"Hi there"` |
| `$` | 문자열/행의 끝 | `end$` | `"the end"` |
| `\d` | 숫자 (0–9) | `\d{3}` | `123`, `007` |
| `\D` | 숫자가 아닌 문자 | `\D+` | `"abc"`, `"--"` |
| `\w` | 단어문자 `[A-Za-z0-9_]` | `\w+` | `"hello"`, `"Python3"` |
| `\W` | 단어문자가 아닌 것 | `\W+` | `"!!"`, `" "` |
| `\s` | 공백 (스페이스, 탭, 개행) | `a\sb` | `"a b"` |
| `\S` | 공백이 아닌 문자 | `\S+` | `"text"`, `"123"` |
| `*` | 0회 이상 반복 | `ab*` | `"a"`, `"ab"`, `"abbb"` |
| `+` | 1회 이상 반복 | `ab+` | `"ab"`, `"abbb"` |
| `?` | 0회 또는 1회 | `ab?` | `"a"`, `"ab"` |
| `{m,n}` | m~n회 반복 | `\d{2,4}` | `99`, `2025` |
| `( )` | 그룹화 / 캡처 | `(ab)+` | `"ab"`, `"abab"` |
| `(?: )` | 비캡처 그룹 | `(?:ab)+` | `"abab"` |
| `|` | OR 선택 | `cat|dog` | `"cat"`, `"dog"` |
| `\b` | 단어 경계 | `\bcat\b` | `"cat"` (단어 단독일 때) |
| `(?i)` | 대소문자 무시 플래그 | `(?i)abc` | `"abc"`, `"ABC"` |

#### 파이썬 정규표현식 함수 요약

| 함수 | 설명 | 예시 코드 | 결과 |
|------|------|-----------|------|
| `re.match(pattern, string)` | 문자열 **처음부터** 패턴 매칭 | `re.match(r"\d+", "123abc")` | `<Match '123'>` |
| `re.search(pattern, string)` | 문자열 전체에서 **처음 매칭되는 패턴** 찾기 | `re.search(r"\d+", "abc123xyz")` | `<Match '123'>` |
| `re.findall(pattern, string)` | **모든 매칭 결과**를 리스트로 반환 | `re.findall(r"\d+", "a12 b34 c56")` | `['12', '34', '56']` |
| `re.finditer(pattern, string)` | 모든 매칭 결과를 **이터레이터(객체)** 로 반환 | `[m.group() for m in re.finditer(r"\d+", "a12 b34")]` | `['12', '34']` |
| `re.sub(pattern, repl, string)` | 패턴을 다른 문자열로 **치환** | `re.sub(r"\d+", "#", "ID123")` | `"ID#"` |
| `re.split(pattern, string)` | 패턴 기준으로 문자열 **분리** | `re.split(r"\s+", "a b   c")` | `['a', 'b', 'c']` |
| `re.fullmatch(pattern, string)` | 문자열 전체가 패턴과 **완전히 일치**할 때 매칭 | `re.fullmatch(r"\d{3}", "123")` | `<Match '123'>` |
| `re.compile(pattern)` | 정규표현식을 객체로 컴파일 (재사용 최적화) | `p = re.compile(r"\d+")`<br>`p.findall("1a2b3")` | `['1','2','3']` |

> ⚠️ `match`는 문자열의 시작 부분만 확인, `search`는 전체 탐색을 수행한다는 점이 중요합니다.

#### 자주 쓰는 패턴 예시

In [ ]:
### 1) 숫자/기호 제거 + 공백 정규화

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "오늘은 2025년 9월 10일!!! 날씨   정말 좋다   ^^"
no_digits = re.sub(r"\d+", "", text)               # 숫자 제거
no_punct  = re.sub(r"[^\w\s가-힣]", " ", no_digits) # 기호 제거(한글/영문/숫자/공백만 남김)
cleaned   = re.sub(r"\s+", " ", no_punct).strip()   # 다중 공백 → 단일 공백
print(cleaned)  # "오늘은 년 월 일 날씨 정말 좋다"

In [ ]:
### 2) 이메일 추출

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문의: admin@example.com, 혹은 support@my-site.co.kr 로 연락주세요."
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)  # ['admin@example.com', 'support@my-site.co.kr']

In [ ]:
### 3) URL 제거 (http/https)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "공식 문서: https://docs.python.org 참고, 우리 블로그 http://example.com/blog 도 봐요."
no_url = re.sub(r"https?://\S+", "", text).strip()  # 정규표현식으로 문자열을 치환합니다.
print(no_url)  # "공식 문서:  참고, 우리 블로그  도 봐요."

In [ ]:
### 4) 한국 전화번호 마스킹

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "연락처: 010-1234-5678 / 02-345-6789"
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)  # 정규표현식으로 문자열을 치환합니다.
print(masked)  # "연락처: 010-****-**** / 02-****-****"

In [ ]:
### 5) 멀티라인에서 행 시작/끝 활용 (`re.M`)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
log = "OK: step1\nERROR: step2 failed\nOK: step3"
errors = re.findall(r"^ERROR:.*$", log, flags=re.M)
print(errors)  # ['ERROR: step2 failed']

In [ ]:
### 6) 탐욕 vs 게으름 (HTML 태그 사이 내용 캡처 예시)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
html = "<p>첫째</p><p>둘째</p>"
greedy = re.findall(r"<p>.*</p>", html)      # 탐욕적: 한 방에 다 먹음
lazy   = re.findall(r"<p>.*?</p>", html)     # 게으름: 가능한 짧게 두 개로 나눔
print(greedy)  # ['<p>첫째</p><p>둘째</p>']
print(lazy)    # ['<p>첫째</p>', '<p>둘째</p>']

> **HTML 파싱은 정규표현식만으로 완벽히 처리하기 어렵습니다.**  
> 웹 페이지처럼 태그 구조가 있는 자료는 Playwright `locator()`처럼 구조를 읽는 도구로 다루는 편이 안정적입니다.


#### 문제 1. 숫자와 기호 제거 + 공백 정규화
문자열에서 **숫자/특수기호를 제거**하고, **다중 공백을 하나로** 바꾼 문자열을 출력하세요.  
한글/영문/공백만 남기도록 하세요.

<details>
<summary>정답 보기</summary>

```python
import re  # 정규표현식을 사용하기 위한 모듈입니다.

text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"
tmp = re.sub(r"\d+", "", text)                         # 숫자를 제거합니다.
tmp = re.sub(r"[^가-힣a-zA-Z\s]", " ", tmp)              # 한글/영문/공백이 아닌 문자를 공백으로 바꿉니다.
result = re.sub(r"\s+", " ", tmp).strip()              # 여러 공백을 하나로 줄이고 양쪽 공백을 제거합니다.
print(result)
# 출력: "정가 원 대박 할인 한정 수량"
```

**문법 설명**
- `import re`: 파이썬 정규표현식 모듈을 불러옵니다.
- `r"..."`: 백슬래시를 정규표현식 문법 그대로 쓰기 위한 원시 문자열입니다.
- `\d+`: 숫자가 1개 이상 연속된 부분을 찾습니다.
- `[^...]`: 대괄호 안의 문자 집합에 포함되지 않는 문자를 찾습니다.
- `가-힣`, `a-zA-Z`, `\s`: 각각 한글 음절, 영문자, 공백 문자를 의미합니다.
- `re.sub(패턴, 바꿀값, 문자열)`: 패턴에 맞는 부분을 다른 문자열로 치환합니다.
- `.strip()`: 문자열 양쪽의 공백을 제거합니다.
</details>


In [ ]:
# 여기에 작성하세요
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"

#### 문제 2. 이메일만 추출하기
문장에서 모든 이메일을 찾아 **리스트**로 반환하세요.

<details>
<summary>정답 보기</summary>

```python
import re  # 정규표현식을 사용하기 위한 모듈입니다.

text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)
# 출력: ['kim.ai@univ.ac.kr', 'sales-team@example.com', 'bug+test@my.io']
```

**문법 설명**
- `re.findall(패턴, 문자열)`: 문자열 전체에서 패턴과 일치하는 모든 값을 리스트로 반환합니다.
- `[a-zA-Z0-9._%+-]+`: 이메일 아이디에 올 수 있는 문자들이 1개 이상 반복되는 부분입니다.
- `@`: 이메일 주소에서 아이디와 도메인을 구분하는 실제 문자입니다.
- `[a-zA-Z0-9.-]+`: 도메인 이름 부분을 찾습니다.
- `\.`: 정규표현식에서 `.`은 임의의 한 문자이므로, 실제 점을 찾을 때는 `\.`처럼 씁니다.
- `[A-Za-z]{2,}`: 마지막 도메인 구간이 영문 2글자 이상인 경우를 찾습니다.
</details>


In [ ]:
# 여기에 작성하세요
text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"

#### 문제 3. URL 제거하기
문장에서 **http/https URL을 모두 제거**하세요. 제거 후 남은 문장의 공백도 정리하세요.

<details>
<summary>정답 보기</summary>

```python
import re  # 정규표현식을 사용하기 위한 모듈입니다.

text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."
no_url = re.sub(r"https?://\S+", "", text)     # http 또는 https로 시작하는 URL을 제거합니다.
no_url = re.sub(r"\s+", " ", no_url).strip()  # 여러 공백을 하나로 줄이고 양쪽 공백을 제거합니다.
print(no_url)
# 출력: "문서: 블로그: 끝."
```

**문법 설명**
- `https?`: `http` 뒤의 `s`가 0번 또는 1번 나올 수 있다는 뜻입니다. 즉 `http`와 `https`를 모두 찾습니다.
- `://`: URL에 들어가는 실제 문자입니다.
- `\S+`: 공백이 아닌 문자가 1개 이상 이어지는 부분을 찾습니다.
- 첫 번째 `re.sub()`는 URL을 빈 문자열로 바꾸고, 두 번째 `re.sub()`는 URL 제거 후 생긴 중복 공백을 정리합니다.
</details>


In [ ]:
# 여기에 작성하세요
text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."

#### 문제 4. 전화번호 마스킹
문장에서 한국식 전화번호(예: `010-1234-5678`, `02-345-6789`)의 **가운데/끝 4자리**를 `*`로 마스킹하세요.

<details>
<summary>정답 보기</summary>

```python
import re  # 정규표현식을 사용하기 위한 모듈입니다.

text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)
print(masked)
# 출력: "문의: 010-****-**** / 대리점: 031-****-**** / 회사: 02-****-****"
```

**문법 설명**
- `\b`: 단어 경계입니다. 전화번호 앞뒤가 다른 단어와 붙어 있는 경우를 줄여 줍니다.
- `(\d{2,3})`: 숫자 2~3자리를 첫 번째 그룹으로 묶습니다. 지역번호 또는 휴대폰 앞자리에 해당합니다.
- `(\d{3,4})`, `(\d{4})`: 가운데 번호와 마지막 번호를 각각 그룹으로 찾습니다.
- `-`: 전화번호 사이의 실제 하이픈 문자입니다.
- `r"\1-****-****"`: 첫 번째 그룹은 그대로 남기고, 뒤 두 그룹은 `****`로 바꿉니다.
</details>


In [ ]:
# 여기에 작성하세요
text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"

### 2.3 데이터 정제 (Cleaning)
2.1에서 수집한 원본 표는 바로 토큰화하지 않고 먼저 정제합니다. 웹에서 가져온 텍스트에는 줄바꿈, 중복 공백, 광고 문구, HTML 흔적, 빈 값, 중복 행이 섞이는 경우가 많습니다.

정제는 **수집한 원본 데이터를 분석 가능한 텍스트 데이터로 바꾸는 단계**입니다.

#### 정제 대상
- 불필요한 특수문자, 구두점  
- HTML 태그와 HTML 엔티티  
- 중복된 공백, 줄바꿈  
- 빈 문자열, 결측치, 중복 행  
- 대소문자 혼재  
- 불용어(stopwords)


####  다양한 예시

1) **특수문자 제거**
```
원문: "안녕??? 오늘 날씨 진짜 좋다~~~^^"
정제: "안녕 오늘 날씨 진짜 좋다"
```

2) **HTML 태그 제거**
```
원문: "<div>이 영화 <b>정말</b> 최고!!!</div>"
정제: "이 영화 정말 최고"
```

3) **중복 공백/개행 제거**
```
원문: "오늘은   점심에    김밥을   먹었다. \n\n 내일도 김밥?"
정제: "오늘은 점심에 김밥을 먹었다. 내일도 김밥?"
```

4) **대소문자 통일**
```
원문: "Apple is Better than apple."
정제(소문자화): "apple is better than apple."
```

5) **불용어 제거** *(전통 ML·IR에서 선택적 / **LLM 파이프라인에선 보통 사용하지 않음**)*  
- **불용어(Stopwords)**: 문장에서 자주 등장하지만 분류·검색 성능에 **한정적으로만** 기여하는 단어 집합.  
  예: 국문 — “나는/그리고/하지만/오늘/에서 …”, 영문 — “the/and/of/to …”  
- 전통 BoW/TF-IDF, IR 인덱싱에서 **차원 축소·노이즈 감소** 목적으로 **선택적** 사용.  
- LLM 파이프라인(사전학습/미세조정/추론)에서는 **보존**하는 것이 일반적.

```
원문: "나는 오늘 점심으로 김밥을 먹었다."
정제(불용어 제거 예 — 전통 ML/IR용): "오늘 점심 김밥 먹었다"
```


👉 이렇게 정제를 통해 **텍스트를 더 단순하고 의미 중심적으로 바꿔야** 이후 단계(토큰화, 임베딩 등)가 효과적으로 작동합니다.

#### 목적에 따라 전처리 전략은 달라집니다

전처리가 항상 좋은 것은 아닙니다. **무엇을 위해 사용하는가**에 따라 달라집니다.

| 목적 | 전처리 필요성 | 이유 |
|------|----------------|------|
| **전통 워드클라우드, TF-IDF 분석** | 높음 | 단순 빈도 기반이므로 노이즈 단어가 시각화를 망침 |
| **전통 ML(감성분석, 분류 등)** | 중간 | 불용어 제거·토큰 정규화가 도움되기도 함 |
| **LLM 파이프라인(사전학습·미세조정·RAG·챗봇)** | 낮음 | LLM은 문맥 기반 모델이므로 불용어·기호도 의미 구조 해석에 필요 |
| **언어학적 분석(구문·어휘 다양성)** | 매우 낮음 | 원문 보존이 핵심 |

즉, **좋은 전처리란, 모든 것을 없애는 것이 아니라 목적에 맞게 적절히 다듬는 것**입니다.

#### 실습: 전처리한 텍스트로 워드클라우드 만들기

워드클라우드는 단어 빈도를 글자 크기로 보여주는 시각화입니다.  
조사, 숫자, 특수문자 같은 노이즈를 정리한 뒤에 만들면 핵심 단어가 훨씬 잘 보입니다.

아래는 같은 방식으로 만들 수 있는 워드클라우드 예시입니다.

<img src="image/wordcloud_example.svg" width="760">



In [ ]:
import re  # 정규표현식을 사용하기 위한 모듈입니다.
from collections import Counter  # 단어 빈도를 세는 도구입니다.

import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import koreanize_matplotlib  # 한글 폰트를 자동으로 설정합니다.
from wordcloud import WordCloud  # 단어 빈도를 워드클라우드로 시각화합니다.

In [ ]:
texts = [
    "배송 지연으로 고객 불만이 증가했습니다. 배송 안내가 더 필요합니다.",
    "환불 요청이 많아 상담 대기 시간이 길어졌습니다.",
    "상품 품질은 좋지만 포장 불량과 배송 지연이 반복됩니다.",
    "빠른 환불 처리와 친절한 상담이 고객 만족을 높였습니다.",
    "배송 상태 알림과 교환 절차 안내를 개선해야 합니다."
]  # 분석할 예시 문장입니다.

In [ ]:
stopwords = {"이", "가", "을", "를", "은", "는", "과", "와", "으로", "더"}  # 제외할 단어입니다.

def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z\s]", " ", text)  # 한글/영문/공백만 남깁니다.
    return re.sub(r"\s+", " ", text).strip()  # 여러 공백을 하나로 정리합니다.

In [ ]:
tokens = []
for text in texts:
    cleaned = clean_text(text)  # 문장별로 노이즈를 제거합니다.
    tokens.extend([word for word in cleaned.split() if word not in stopwords and len(word) > 1])  # 의미 단어만 남깁니다.

In [ ]:
freq = Counter(tokens)  # 단어별 등장 횟수를 계산합니다.
freq.most_common(10)  # 가장 자주 나온 단어를 확인합니다.

In [ ]:
wc = WordCloud(
    font_path=koreanize_matplotlib.get_font_ttf_path(),  # 워드클라우드 한글 표시용 폰트입니다.
    width=900,  # 워드클라우드 이미지의 가로 크기입니다.
    height=450,  # 워드클라우드 이미지의 세로 크기입니다.
    background_color="white",  # 배경색입니다. 예: "white", "black"
    colormap="viridis"  # 단어 색상 팔레트입니다. 예: "viridis", "plasma", "tab10"
).generate_from_frequencies(freq)  # 단어 빈도로 워드클라우드를 만듭니다.

In [ ]:

plt.figure(figsize=(10, 5))  # 그래프 크기와 도화지를 설정합니다.
plt.imshow(wc, interpolation="bilinear")  # 워드클라우드 이미지를 표시합니다.
plt.axis("off")  # 축을 숨깁니다.
plt.show()  # 그래프를 화면에 출력합니다.


### 실무형 EDA: 수집된 뉴스 데이터 살펴보기

워드클라우드를 만들기 전에 수집된 텍스트 데이터의 규모와 품질을 먼저 확인합니다.  
제목과 본문의 글자 수, 결측치, 언론사와 기자명 분포를 보면 어떤 데이터가 많이 들어왔는지 빠르게 파악할 수 있습니다.


In [ ]:
import pandas as pd  # CSV 데이터를 표 형태로 읽기 위한 도구입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import koreanize_matplotlib  # matplotlib 그래프에서 한글이 깨지지 않도록 설정합니다.

news = pd.read_csv("naver_economy_news.csv")  # CSV 파일을 DataFrame으로 읽습니다.

print("행 개수:", len(news))  # 수집된 기사 개수를 확인합니다.
print("열 개수:", len(news.columns))  # 데이터에 들어 있는 컬럼 개수를 확인합니다.
print("컬럼 목록:", news.columns.tolist())  # 어떤 정보가 수집되었는지 컬럼명으로 확인합니다.

news.head()  # 데이터 앞부분을 직접 눈으로 확인합니다.

#### 1. 결측치 확인

결측치는 비어 있는 값입니다. 텍스트 분석 전에는 제목, 본문, 언론사, 기자명처럼 중요한 컬럼에 비어 있는 값이 있는지 먼저 확인합니다.


In [ ]:
text_columns = ["title", "press", "journalist", "time", "body", "url"]  # 이번 EDA에서 확인할 주요 컬럼입니다.

missing_df = news[text_columns].isna().sum().reset_index()  # 컬럼별 결측치 개수를 계산합니다.
missing_df.columns = ["column", "missing_count"]  # 보기 쉬운 컬럼명으로 바꿉니다.

missing_df  # 결측치가 많은 컬럼이 있는지 확인합니다.

#### 2. 분석용 컬럼 정리

글자 수를 계산하기 전에 빈 값은 간단히 채워 둡니다. 그리고 제목과 본문의 길이를 새 컬럼으로 만듭니다.


In [ ]:
news["title"] = news["title"].fillna("").astype(str)  # 제목 결측치는 빈 문자열로 처리합니다.
news["body"] = news["body"].fillna("").astype(str)  # 본문 결측치는 빈 문자열로 처리합니다.
news["press"] = news["press"].fillna("알 수 없음").astype(str)  # 언론사 결측치는 표시용 값으로 채웁니다.
news["journalist"] = news["journalist"].fillna("알 수 없음").astype(str)  # 기자명 결측치는 표시용 값으로 채웁니다.

news["title_length"] = news["title"].str.len()  # 제목 글자 수를 계산합니다.
news["body_length"] = news["body"].str.len()  # 본문 글자 수를 계산합니다.

news[["title", "press", "journalist", "time", "title_length", "body_length"]].head()  # 정리된 컬럼을 확인합니다.


#### 3. 기본 요약 지표 확인

전체 기사 수, 언론사 수, 기자 수, 제목과 본문의 평균 길이를 표로 확인합니다.


In [ ]:
eda_summary = pd.DataFrame({
    "지표": [
        "전체 기사 수",
        "언론사 수",
        "기자 수",
        "제목 글자 수 평균",
        "본문 글자 수 평균",
        "본문 글자 수 중앙값",
    ],
    "값": [
        len(news),
        news["press"].nunique(),
        news["journalist"].nunique(),
        round(news["title_length"].mean(), 1),
        round(news["body_length"].mean(), 1),
        round(news["body_length"].median(), 1),
    ],
})  # 실무에서 먼저 보는 기본 요약 지표를 만듭니다.

eda_summary  # 요약 지표를 표로 확인합니다.


#### 4. 본문 글자 수 히스토그램

히스토그램은 값의 분포를 보는 그래프입니다. 여기서는 기사 본문 길이가 어느 구간에 많이 몰려 있는지 확인합니다.


In [ ]:
plt.figure(figsize=(8, 4))  # 그래프 크기를 정합니다.
plt.hist(news["body_length"], bins=12, color="steelblue", edgecolor="white")  # 본문 글자 수 분포를 그립니다.
plt.title("본문 글자 수 분포")  # 그래프 제목입니다.
plt.xlabel("본문 글자 수")  # x축은 본문 길이입니다.
plt.ylabel("기사 수")  # y축은 해당 길이 구간의 기사 수입니다.
plt.show()  # 그래프를 화면에 출력합니다.

#### 5. 언론사별 기사 수

어떤 언론사의 기사가 많이 수집되었는지 확인합니다. 특정 언론사가 많이 들어오면 워드클라우드 결과도 그 언론사의 기사 내용에 영향을 받을 수 있습니다.


In [ ]:
press_counts = news["press"].value_counts().head(10)  # 기사 수가 많은 상위 10개 언론사를 구합니다.

press_counts_df = press_counts.reset_index()  # Series를 표 형태로 바꿉니다.
press_counts_df.columns = ["press", "article_count"]  # 컬럼명을 보기 쉽게 바꿉니다.

press_counts_df  # 상위 언론사별 기사 수를 표로 확인합니다.

In [ ]:
press_counts_for_plot = press_counts.sort_values()  # 가로 막대그래프가 보기 좋도록 작은 값부터 정렬합니다.

plt.figure(figsize=(8, 5))  # 그래프 크기를 정합니다.
plt.barh(press_counts_for_plot.index, press_counts_for_plot.values, color="darkorange")  # 언론사별 기사 수를 그립니다.
plt.title("상위 언론사별 기사 수")  # 그래프 제목입니다.
plt.xlabel("기사 수")  # x축은 기사 수입니다.
plt.ylabel("언론사")  # y축은 언론사 이름입니다.
plt.show()  # 그래프를 화면에 출력합니다.


#### 6. 기자명별 기사 수

기자명도 함께 확인합니다. 같은 기자의 기사가 많이 수집되었는지 보면 데이터 편향을 간단히 점검할 수 있습니다.


In [ ]:
journalist_counts = news["journalist"].value_counts().head(10).reset_index()  # 기사 수가 많은 상위 10명 기자를 구합니다.
journalist_counts.columns = ["journalist", "article_count"]  # 컬럼명을 보기 쉽게 바꿉니다.

journalist_counts  # 상위 기자명별 기사 수를 표로 확인합니다.


### 실무형 워드클라우드 연습

이번에는 `naver_economy_news.csv`에 저장된 네이버 경제 뉴스 본문을 모아 워드클라우드를 만들어 봅니다.  
`stopword_list`에 기본 예시 단어 2개를 넣어 두었습니다. 자주 등장하지만 핵심 의미가 약한 단어를 같은 위치에 직접 추가해 보세요.


In [ ]:
import re  # 정규표현식을 사용하기 위한 모듈입니다.
from collections import Counter  # 단어 빈도를 세는 도구입니다.
from pathlib import Path  # 파일 경로를 다루기 위한 도구입니다.

import pandas as pd  # CSV 데이터를 표 형태로 읽기 위한 도구입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import koreanize_matplotlib  # 한글 폰트를 자동으로 설정합니다.
from wordcloud import WordCloud  # 단어 빈도를 워드클라우드로 시각화합니다.

csv_path = Path("naver_economy_news.csv")  # 오늘 수집한 네이버 경제 뉴스 파일입니다.
news = pd.read_csv(csv_path)  # CSV 파일을 DataFrame으로 읽습니다.
news_bodies = news["body"].dropna().astype(str)  # 뉴스 본문 컬럼만 사용합니다.

In [ ]:
stopword_list = [
    "기자",
    "제공",
    # 여기에 제외하고 싶은 단어를 직접 추가하세요.
]
stopwords = set(stopword_list)  # 빠른 검색을 위해 리스트를 set으로 바꿉니다.

In [ ]:
def clean_news_text(text):
    text = re.sub(r"[^가-힣a-zA-Z\s]", " ", text)  # 한글/영문/공백만 남깁니다.
    return re.sub(r"\s+", " ", text).strip()  # 여러 공백을 하나로 정리합니다.

In [ ]:
tokens = []
for body in news_bodies:
    cleaned = clean_news_text(body)  # 기사 본문별로 노이즈를 제거합니다.
    tokens.extend([
        word for word in cleaned.split()
        if word not in stopwords and len(word) > 1
    ])  # 의미 단어 후보만 남깁니다.
tokens[:5]

In [ ]:
freq = Counter(tokens)  # 단어별 등장 횟수를 계산합니다.
freq

In [ ]:

top_keywords = pd.DataFrame(freq.most_common(30), columns=["keyword", "count"])  # 상위 키워드를 표로 확인합니다.

display(top_keywords)  # 불용어로 뺄 단어를 찾기 위해 먼저 빈도표를 확인합니다.

wc = WordCloud(
    font_path=koreanize_matplotlib.get_font_ttf_path(),  # 워드클라우드 한글 표시용 폰트입니다.
    width=1000,  # 워드클라우드 이미지의 가로 크기입니다.
    height=500,  # 워드클라우드 이미지의 세로 크기입니다.
    background_color="white",  # 배경색입니다.
    colormap="tab10",  # 경제 뉴스 키워드가 또렷하게 보이도록 색상 팔레트를 지정합니다.
    max_words=120,  # 워드클라우드에 표시할 최대 단어 수입니다.
    random_state=42
).generate_from_frequencies(freq)  # 오늘 뉴스 본문 단어 빈도로 워드클라우드를 만듭니다.

plt.figure(figsize=(12, 6))  # 그래프 크기와 도화지를 설정합니다.
plt.imshow(wc, interpolation="bilinear")  # 워드클라우드 이미지를 표시합니다.
plt.title("오늘 경제 뉴스 핵심 키워드", fontsize=16, pad=16)  # 오늘 뉴스의 핵심을 요약하는 제목입니다.
plt.axis("off")  # 축을 숨깁니다.
plt.show()  # 그래프를 화면에 출력합니다.


#### 문제 5. 대소문자 통일
문장 `"Machine Learning is FUN and Useful."`을 모두 소문자로 바꿔보세요.  

<details>
<summary>정답 보기</summary>

```python
text = "Machine Learning is FUN and Useful."  # 실습할 문자열을 준비합니다.
result = text.lower()  # 모든 영문자를 소문자로 바꾼 새 문자열을 만듭니다.
print(result)
# 출력: "machine learning is fun and useful."
```

**문법 설명**
- `text.lower()`: 문자열 안의 영문 대문자를 소문자로 바꿉니다.
- 문자열 메서드는 원본 문자열을 직접 바꾸지 않고 새 문자열을 반환합니다.
- `result = ...`: 변환된 값을 다시 사용할 수 있도록 변수에 저장합니다.
- `print(result)`: 변수에 저장된 결과를 화면에 출력합니다.
</details>


In [ ]:
# 여기에 작성하세요
text = "Machine Learning is FUN and Useful."

#### 문제 6. 불용어 제거
문장 `"나는 오늘 아침에 학교에 갔다."`에서 불용어 `["나는", "오늘", "에"]`를 제거해보세요.  

<details>
<summary>정답 보기</summary>

```python
text = "나는 오늘 아침에 학교에 갔다."  # 실습할 문자열을 준비합니다.
stopwords = ["나는", "오늘", "에"]  # 제거할 불용어 목록입니다.

for word in stopwords:
    text = text.replace(word, " ")  # 현재 불용어를 공백으로 바꿉니다.

result = " ".join(text.split())  # 여러 공백을 하나로 정리합니다.
print(result)
# 출력: "아침 학교 갔다."
```

**문법 설명**
- `stopwords = [...]`: 여러 값을 순서대로 담는 리스트입니다.
- `for word in stopwords`: 리스트의 값을 하나씩 꺼내 같은 작업을 반복합니다.
- `text.replace(찾을문자열, 바꿀문자열)`: 문자열에서 특정 부분을 찾아 다른 문자열로 바꿉니다.
- `text.split()`: 공백 기준으로 문자열을 나누며, 연속된 공백은 자동으로 무시합니다.
- `" ".join(...)`: 나뉜 단어들을 공백 하나로 다시 이어 붙입니다.
- 이 방식은 단순 문자열 치환이므로, 실제 한국어 분석에서는 토큰화나 형태소 분석 기준으로 불용어를 제거하는 편이 더 안정적입니다.
</details>


In [ ]:
# 여기에 작성하세요
text = "나는 오늘 아침에 학교에 갔다."
stopwords = ["나는", "오늘", "에"]

### 2.4 토큰화(Tokenization)

토큰화(Tokenization)는 긴 텍스트를 단어·형태소처럼 **분석하기 좋은 단위**로 나누는 과정입니다.  
워드클라우드나 키워드 빈도 분석에서는 어떤 단위로 자르느냐에 따라 결과가 달라집니다.

예를 들어 리뷰 문장 `배송은 빠른데 포장이 아쉬워요`는 다음처럼 볼 수 있습니다.

- 띄어쓰기 기준: `배송은`, `빠른데`, `포장이`, `아쉬워요`
- 의미 중심 기준: `배송`, `빠르다`, `포장`, `아쉽다`

여기서는 핵심만 확인합니다. 기본은 `split()`으로 나누고, 한국어 키워드는 KoNLPy의 `Okt.nouns()`로 명사를 뽑아봅니다.


#### 토큰화 기준 간단히 보기

텍스트마이닝에서는 목적에 따라 토큰 단위를 다르게 잡습니다.

| 기준 | 예시 | 사용 상황 |
|------|------|-----------|
| 띄어쓰기 | `배송은`, `빠른데` | 빠르게 문장을 나눠볼 때 |
| 명사 추출 | `배송`, `포장`, `환불` | 키워드와 이슈를 볼 때 |
| 형태소 분석 | `빠르다`, `아쉽다` | 감성·상태 표현까지 보고 싶을 때 |

처음에는 띄어쓰기 기준으로 시작하고, 한국어 키워드를 더 깔끔하게 보고 싶을 때 명사 추출을 사용하면 됩니다.


#### 한국어 형태소 분석: KoNLPy

한국어는 조사와 어미가 단어에 붙어서 의미를 만듭니다.  
띄어쓰기만 기준으로 자르면 `고객이`, `고객은`, `고객에게`가 서로 다른 단어처럼 남을 수 있습니다.

KoNLPy는 한국어 형태소 분석기를 파이썬에서 사용할 수 있게 해주는 라이브러리입니다.  
아래 예제에서는 `Okt` 분석기로 형태소, 명사, 품사를 간단히 확인합니다.

> **설치 안내**  
> Windows 로컬 설치가 어렵다면 수업 실습은 Google Colab에서 진행하는 것을 권장합니다.


In [ ]:
!pip install konlpy

In [ ]:
from konlpy.tag import Okt

okt = Okt()

text = "고객이 배송 지연으로 환불을 요청했습니다."

print("형태소:", okt.morphs(text))  # 형태소 단위로 나눈 결과를 출력합니다.
print("명사:", okt.nouns(text))  # 명사만 추출한 결과를 출력합니다.
print("품사:", okt.pos(text))  # 형태소와 품사 태그를 함께 출력합니다.


#### 문제 7. KoNLPy로 명사 추출
문장 `"품질 점검 중 센서 오류가 반복적으로 발생했습니다."`에서 명사만 추출하세요.  

<details>
<summary>정답 보기</summary>

```python
from konlpy.tag import Okt

okt = Okt()  # Okt 형태소 분석기 객체를 만듭니다.
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."
nouns = okt.nouns(sentence)  # 문장에서 명사 후보만 추출합니다.
print(nouns)
# 출력 예시: ['품질', '점검', '중', '센서', '오류']
```

**문법 설명**
- `from konlpy.tag import Okt`: KoNLPy에서 Okt 형태소 분석기 클래스를 가져옵니다.
- `okt = Okt()`: 분석기를 사용할 수 있도록 객체를 생성합니다.
- `okt.nouns(sentence)`: 입력 문장에서 명사 후보를 리스트로 반환합니다.
- 분석기와 버전에 따라 추출 결과가 조금 달라질 수 있습니다.
</details>


In [ ]:
# 여기에 정답을 작성하세요
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."


#### 문제 8. 단어 단위 토큰화
문장 `"오늘은 자연어 처리를 공부한다."`를 **띄어쓰기 기준**으로 토큰화하세요.  

<details>
<summary>정답 보기</summary>

```python
sentence = "오늘은 자연어 처리를 공부한다."  # 분석할 문장을 준비합니다.
tokens = sentence.split()  # 공백 기준으로 문자열을 나눕니다.
print(tokens)
# 출력: ['오늘은', '자연어', '처리를', '공부한다.']
```

**문법 설명**
- `sentence.split()`: 인자를 넣지 않으면 공백, 탭, 줄바꿈 같은 공백 문자를 기준으로 문자열을 나눕니다.
- 반환값은 문자열 조각들이 담긴 리스트입니다.
- 마침표 같은 문장부호는 자동으로 제거되지 않습니다. 그래서 `공부한다.`처럼 마침표가 붙은 채로 남습니다.
</details>


In [ ]:
# 여기에 정답을 작성하세요
sentence = "오늘은 자연어 처리를 공부한다."

### 2.5 미니 실습: 리뷰 키워드 확인

토큰화와 형태소 분석을 활용하면 여러 문장에서 자주 등장하는 키워드를 빠르게 확인할 수 있습니다.  
아래 예시는 리뷰 문장에서 명사만 모아 어떤 이슈가 많이 등장하는지 보는 간단한 텍스트마이닝 흐름입니다.


In [ ]:
from collections import Counter  # 단어별 등장 횟수를 세기 위해 사용하는 도구입니다.

# 분석할 리뷰 예시입니다. 실제 업무에서는 쇼핑몰 리뷰, 상담 이력, 설문 응답 등이 여기에 들어갈 수 있습니다.
reviews = [
    "배송은 빨랐지만 포장이 조금 아쉬웠어요.",
    "제품 품질이 좋고 가격도 만족스럽습니다.",
    "배송 지연 때문에 고객센터에 문의했습니다.",
    "포장 상태가 좋아서 파손 없이 받았습니다.",
    "가격 대비 품질은 좋은데 배송 안내가 부족했습니다.",
]

# 앞에서 만든 okt = Okt() 객체를 사용해 각 리뷰에서 명사만 추출합니다.
# nouns 리스트에는 모든 리뷰에서 뽑은 명사를 한곳에 모아 둡니다.
nouns = []

# 리뷰를 한 문장씩 꺼내서 반복 처리합니다.
for review in reviews:
    # okt.nouns(review)는 문장에서 명사 후보만 뽑아 리스트로 돌려줍니다.
    review_nouns = okt.nouns(review)

    # 한 글자 단어는 의미 있는 키워드로 보기 어려운 경우가 많아 제외합니다.
    for word in review_nouns:
        if len(word) >= 2:
            nouns.append(word)

# Counter는 같은 단어가 몇 번 등장했는지 자동으로 세어 줍니다.
keyword_counts = Counter(nouns)

# 가장 많이 등장한 단어 10개를 (단어, 등장 횟수) 형태로 확인합니다.
keyword_counts.most_common(10)


#### 정리

텍스트마이닝에서는 원문을 바로 모델에 넣기보다 분석하기 쉬운 형태로 준비합니다.

- 수집: 분석할 텍스트를 웹, 파일, 설문 등에서 확보합니다.
- 정제: 불필요한 문자, HTML 태그, 중복 공백 같은 노이즈를 줄입니다.
- 토큰화: 문장을 단어, 형태소, n-gram처럼 분석 가능한 단위로 나눕니다.


### 2.7 추가 실습: 나만의 워드클라우드 만들기

이번 실습의 목표는 **직접 수집한 텍스트 데이터로 EDA를 수행한 뒤, 자신만의 워드클라우드를 만들어 노션에 업로드**하는 것입니다.

#### 진행 순서

1. 관심 있는 공개 웹페이지, 뉴스, 리뷰, 게시글 등에서 텍스트 데이터를 직접 수집합니다.
2. 수집한 데이터를 CSV 파일로 저장합니다. 최소한 텍스트 컬럼 1개와 출처 또는 링크 컬럼 1개를 포함합니다.
3. 수집 데이터의 기본 EDA를 수행합니다.
   - 데이터 행/열 개수 확인
   - 결측치와 중복 데이터 확인
   - 텍스트 길이 분포 확인
   - 출처, 카테고리, 작성자 등 수집 데이터의 주요 분포 확인
4. 분석에 사용할 텍스트 컬럼을 정하고, 불필요한 기호·숫자·공백·불용어를 정리합니다.
5. 단어 빈도를 계산한 뒤 워드클라우드를 생성합니다.
6. 워드클라우드에서 크게 보이는 단어를 바탕으로 데이터의 특징을 3~5문장으로 해석합니다.

#### 노션 업로드 항목

- 수집한 데이터 주제와 출처
- 수집 데이터의 행 개수와 주요 컬럼 설명
- EDA 결과 화면 또는 요약 표
- 직접 만든 워드클라우드 이미지
- 워드클라우드에서 확인한 핵심 키워드와 간단한 해석

최종 제출물은 **본인이 직접 수집한 데이터로 만든 워드클라우드가 포함된 노션 페이지**입니다.
